# RSAM notebook

In this tutorial, we will explore Real-time Seismic Amplitude Measurement (RSAM) data. 

RSAM is, by definition, computed on raw seismic data, so we can also think of it as "Raw" Seismic Amplitude Measurement, to distinguish from similar measurements we will make later on velocity and displacement seismograms.

## 1. Original RSAM system

The RSAM system was built around a 8-bit analog-to-digital-converter PC card: software was too slow in those days. Components of the original RSAM system were:

<font color='yellow'>
<ol>
<li>Real-time bar graphs: showing average seismic amplitudes over last 2.56 s, 1 minute, and 10 minutes</li>
<li><b>1 minute and 10 minute mean signal amplitudes, logged to binary files. This is what most volcano-seismologists today think of as "RSAM data"!</b></li>
<li>"RSAM events": created by a simple STA/LTA detector running on each channel (NSLC)</li>
<li>Multi-station event (e.g. earthquake) and tremor alarm systems</li>
<li>Trends in RSAM data and other datasets (e.g. earthquake counts, tiltmeter data, gas flux, deformation, etc.) could be visualized with another software package called "BOB"</li>
</ol></font>

<table border=1><tr><td><img width=100% src="../mess2024/images/rsam.png" ></td><td>RSAM barcharts from Glowworm running in Montserrat in 2002. Tom Murray wrote the original RSAM system in 1985 and the GlowWorm system c. 1998 to provide volcano-monitoring extensions to Earthworm.</td></table></tr></table>

<table border=1><tr><td><img width=100% src="../mess2024/images/EndoMurray1991fig7.png" ></td><td>BOB plot. Fig 7 from Endo & Murray (1991). Top panel shows RSAM event rate at closest station to Pinatubo. Bottom 3 panels show RSAM data from stations at increasing distances. 30 days of data are show</td></table></tr></table>

In the figure above, 30 days of RSAM data are shown for three seismic stations. Loading and plotting 30 days of raw seismic data takes a while, but 1-minute RSAM data downsamples the raw seismic data by a factor of 6,000 (assuming a 100 Hz sampling rate), so long RSAM timeseries (hours, days, weeks, months, etc.) can be quickly loaded and plotted.

Reference:
- Endo, E.T., Murray, T. Real-time Seismic Amplitude Measurement (RSAM): a volcano monitoring and prediction tool. Bull Volcanol 53, 533–545 (1991).__[https://doi.org/10.1007/BF00298154](../../pdf/RSAM_EndoMurray1991.pdf)__

## 2. The RSAM class

We will exploit the RSAM class here (class in the Object-Oriented sense) which is in flovopy/processing/sam.py (a "Python module"). One of the methods of the RSAM class is to read the binary files written by the original RSAM system, as we will see.

More importantly, the RSAM class is a convenient way of downsampling (raw) seismic data. The original RSAM system used 2.56s, 60s, and 600s. I prefer to use 2.56s for events, and 60s for continuous data. For each time window, the RSAM class computes the following features:

- mean amplitude
- median amplitude
- max amplitude
- std (same as rms after detrending) amplitude
- mean amplitude in "VT band"
- mean amplitude in "LP band"
- mean amplitude in "VLP band"
- base-2 logarithm of the ratio of VT to LP band amplitudes (frequency ratio ...)

These are all features can be quickly computed because they can be done in the time domain. 

We can process the data in various ways. Some of these, e.g. using `select()`, `trim()`, `plot()`, should be familiar from ObsPy `Stream` objects. Others such as `downsample()` are not.

## 3. Frequency ratio
One of the metrics computed by the RSAM class is the "Frequency Ratio", which is a base-2 logarithm of the amplitude ratio of the VT frequency band versus the LP frequency band:

\begin{align}
fratio & = log_{2} \frac {A_{VT}}{A_{LP}} \\
\end{align}

                                                                        (Rodgers et al., 2016)

We use the following definitions of the VT and LP bands:

<table border=1>
    <tr><td>Class</td><td>Frequency Band (Hz)</td></tr>
    <tr><td>LP</td><td>0.8 - 4.0</td></tr>
    <tr><td>VT</td><td>4.0 - 18.0</td></tr>
</table>


## 4. Simple example

Here is a minimal example of computing RSAM data from an ObsPy Stream object. The data come from vertical components of stations REF & RSO at Redoubt Volcano in Alaska from 2009/03/19 to 2009/03/23.

### 4.1 Import path to the Samba share
First, make sure you have mounted the Samba share from the server "hal9000". Do you remember how we did this in week 8?

Next, let's set the path to the Samba mounted share from the server 'hal9000', as we did in week8:

In [ ]:
import sys
sys.path.append('../week8')
from set_samba_data_root import DATA_ROOT
print(DATA_ROOT)

## 4.2 Loading data from the SDS archive

In [ ]:
from flovopy.processing.sam import RSAM # RSAM class from flovopy
from obspy.clients.filesystem.sds import Client # for reading SDS archives
from obspy.core import UTCDateTime, Stream

SDS_ROOT = DATA_ROOT / "SDS_Alaska"   # <-- SDS archive made from AVO and AEC data
NET = "AV"
STATIONS = ['REF', 'RSO'] # a list of stations
LOC = ""
CHA = "EHZ"   # vertical channels only

# Date range (inclusive start, exclusive end)
t0 = UTCDateTime("2009-03-19T00:00:00")
t1 = UTCDateTime("2009-03-23T07:00:00")

print(f"SDS_ROOT: {SDS_ROOT}")
print(f"Date range: {t0} to {t1}")

# Create SDS client
client = Client(str(SDS_ROOT))

st = Stream() # create an empty Stream object
for STA in STATIONS: # Loop over all stations (well just 2)
    temp_st = client.get_waveforms(NET, STA, LOC, CHA, t0, t1) # <-- Read data from SDS
    temp_st.merge() # otherwise we get lots of Trace objects for each SEED id
    for tr in temp_st: # Loop over each Trace in this temporary Stream
        st.append(tr) # Append this Trace to the master Stream
print(st)
st.plot(equal_scale=False);

You can't see much here because both traces have spikes. They are both peak at around 250,000,000 counts! Both RSO and REF are short-period seismometers with analog FM telemetry. That means the data are digitized at the observatory. So this is probably some digitizer noise from interference. 

## 4.3 What is FM radio?

In FM (frequency modulation) radio telemetry, a measured signal (like seismic amplitude) is used to vary the frequency of a high-frequency carrier wave, which is a stable radio signal (a sine wave, e.g. at 400 MHz) that can travel long distances. The original data signal itself is too low-frequency to transmit efficiently (and would require massive antennas), so it is encoded onto the carrier through modulation—in FM, this means small changes in frequency represent the signal’s variations. At the receiving end, a demodulator extracts (recovers) the original signal by tracking those frequency changes and converting them back into the measured data.

Antenna size is fundamentally tied to the wavelength of the signal, which is inversely proportional to frequency ($\lambda = c/f$). Efficient antennas are typically a fraction of the wavelength (often a quarter-wave), so higher-frequency radio signals use conveniently small antennas. For example, a 400 MHz signal has a wavelength of ~0.75 m, so a quarter-wave antenna is only ~19 cm long. In contrast, seismic signals are extremely low frequency (e.g., 1 Hz), corresponding to wavelengths of hundreds of thousands of kilometers—meaning a quarter-wave antenna would be tens of thousands of kilometers long. This is why seismic data cannot be transmitted directly at its native frequency and instead must be modulated onto a high-frequency carrier for practical radio telemetry.

### Here is a little demo of FM 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from obspy import Trace

def plot_fm_like_wikipedia(
    tr: Trace,
    carrier_hz: float = 100.0,
    peak_deviation_hz: float = 20.0,
    duration: float = 2.0,
    sim_fs: float = 5000.0,
):
    """
    Plot a real FM waveform like the classic AM/FM modulation diagrams.

    Parameters
    ----------
    tr : obspy.Trace
        Input trace used as the message signal.
    carrier_hz : float
        Carrier frequency for the visible demo.
    peak_deviation_hz : float
        Peak frequency deviation when the normalized message reaches ±1.
    duration : float
        Seconds of input trace to use.
    sim_fs : float
        Internal sample rate for plotting the carrier waveform cleanly.
    """
    fs_in = float(tr.stats.sampling_rate)
    if fs_in <= 0:
        raise ValueError("Trace must have a positive sampling rate.")
    if sim_fs <= 10 * carrier_hz:
        raise ValueError("sim_fs should be much larger than carrier_hz for a clean plot.")

    # --- input message ---
    n_in = min(len(tr.data), int(duration * fs_in))
    x = tr.data[:n_in].astype(float)
    t_in = np.arange(n_in) / fs_in

    # normalize message to [-1, 1]
    x = x - np.mean(x)
    peak = np.max(np.abs(x))
    if not np.isfinite(peak) or peak == 0:
        raise ValueError("Trace has no usable amplitude variation.")
    m_in = x / peak

    # --- interpolate message to a fine grid for visible carrier plotting ---
    n_sim = int(duration * sim_fs)
    t = np.arange(n_sim) / sim_fs
    m = np.interp(t, t_in, m_in)

    # --- actual FM waveform ---
    # instantaneous frequency
    f_inst = carrier_hz + peak_deviation_hz * m

    # phase = integral of instantaneous frequency
    phase = 2 * np.pi * np.cumsum(f_inst) / sim_fs

    # real FM signal: constant amplitude, variable cycle spacing
    s_fm = np.cos(phase)

    # --- plots ---

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)

    axes[0].plot(t, m, linewidth=1.2)
    axes[0].set_ylabel("Message")
    axes[0].set_title(f"Input signal ({tr.id})")

    axes[1].plot(t, f_inst, linewidth=1.2)
    axes[1].axhline(carrier_hz, linestyle="--", alpha=0.7)
    axes[1].set_ylabel("Hz")
    axes[1].set_title("Instantaneous frequency")

    axes[2].plot(t, s_fm, linewidth=1.0)
    axes[2].set_ylabel("FM signal")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_title(
        f"FM radio signal"
    )

    fig.tight_layout()
    plt.show()

    return {
        "t": t,
        "message": m,
        "f_inst": f_inst,
        "fm_signal": s_fm,
    }

duration = 2
tr = st[0].copy().trim(starttime=t0+2, endtime=t0 + 2 + duration)  # Use first 5 seconds of the first Trace
result = plot_fm_like_wikipedia(
    tr,
    carrier_hz=50.0,
    peak_deviation_hz=30.0,
    duration=duration,
    sim_fs=5000.0,
)



The panels above show: (top) original analog signal, (middle) mapping from amplitude to frequency. The carrier wave here has a frequency of 50 Hz, and can vary by up to 30 Hz (20 Hz - 80 Hz) to cover the range from the most -ve to the most +ve amplitude in the signal. (bottom) the frequency modulated radio signal. This looks like a spring, where compressions (up to 80 Hz) indicate high amplitudes, and rarefactions (down to 20 Hz) indicate low amplitudes. I reality the carrier wave would be more like 400 MHz, but we wouldn't be able to see that vary on a plot, so we used just 50 Hz.

A demodulator at the observatory reverses (decodes) the encoding, and recovers the original signal. In reality, each seismic station in FM telemetry operates in a different FM band to avoid interference, and discriminators (tuned radios essentially) are used to "listen" to each incoming signal, and send it to a demodulator.


## 4.3 Despiking seismic data

OK, so following that aside on FM radio, let's get back to our problem - SPIKES!

How can we eliminate high amplitude spikes? Numpy has a clip() function. Let's try that. But what is the actual range of good data? We could just clip at say twice the 99% percentile. Let's try that!

In [ ]:
import numpy as np

def clip_trace(trace, percentile=99.9):
    '''
    Clip the data in this Trace to +/- 2x the given percentile of the absolute value of the data.
    '''
    x = trace.data
    clip_level = np.nanpercentile(np.abs(x), percentile) * 2
    print(f"Clipping {trace.id} at {clip_level:.2f}")
    trace.data = np.clip(trace.data, -clip_level, clip_level)

def clip_stream(stream, percentile=99.9):
    for tr in stream:
        clip_trace(tr, percentile=percentile)

st2 = st.copy()
clip_stream(st2, percentile=99.9)
st2.plot(equal_scale=False);

So now we can see the range of the REAL signal. Notice that data are still squared off. Why? That is likely because the analog FM telemetry itself had limited dynamic range. FM, or Frequency Modulated, radio signals are broadcast in a certain radio frequency band. So they have a minimum and maximum frequency, which correspond to a minimum and maximum amplitude. So the FM telemetry likely clipped the signal coming from the short-period seismometer, squaring off the tops and bottoms of the signal.

It looks like we have continuous square waves on RSO from late on March 20, 2009. Let's zoom in to see if that is true:

In [ ]:
st2.plot(equal_scale=False, starttime=t0+86400*2.2, endtime=t0+86400*2.3);

So by zooming in we can see there is actually plenty of usable signal.

So let's now compute some RSAM data!

Let's see what helpful information we can get about the RSAM class:

In [ ]:
help(RSAM)

An RSAM object can be created by passing an ObsPy Stream object as an input parameter, and setting a sampling_interval (default: 60s). 

By default, the Stream object will be bandpass filtered from 0.5-18.0 Hz (the `filter` parameter) before computing RSAM. However, some other filters are defined for the VLP (0.02-0.2 Hz), LP (0.5-4.0 Hz), and VT (4.0-18.0 Hz) bands using `bands`, which is a Python dictionary. The frequency ratio is computed from the VT and LP bands, so override these definitions if they should be set differently for your dataset.

Internally, RSAM data for each ObsPy Trace object in the Stream is held in a pandas DataFrame.

In [ ]:
rsamObj = RSAM(stream=st2, sampling_interval=60) # 60-s sampling interval
print(str(rsamObj))
rsamObj.plot()

In the present example, the Stream only contains 2 Traces and we can view the dataframe with:

In [ ]:
print(rsamObj.dataframes)


In [ ]:
for tr in st2:
    print(f'RSAM dataframe for {tr.id}')
    print(rsamObj.dataframes[tr.id])

The 'time' column is in Unix epoch seconds (since 1970-01-01 00:00:00) (<em>I'll probably change this to Python datetime...</em>)

The original RSAM system calculated the mean signal amplitude at sample intervals of 2.56s, 60s, and 600s, as shown in the bar graph above. However, it is cheap and fast today to compute and store other metrics for each sample interval too, so RSAM objects also contain the min, max, median, and rms amplitude of each sample interval. (The aforementioned VLP, LP, and VT bands are computed just with the mean). A full list of the available metrics can be gotten like this:

In [ ]:
print(rsamObj.get_metrics())

Specific metrics can be plotted:

In [ ]:
rsamObj.plot(metrics=['mean','median','rms','max'])

For tremor analysis, I prefer to use the median, because it isn't biased by outliers the way the mean, rms, and max are. On the other hand, if my focus is to see the size of the largest events, it might be best to plot the max.

The way we have plotted the RSAM object so far calls the to_stream() method, so each RSAM DataFrame is converted into an ObsPy Trace before plotting. However, we can also plot the DataFrame directly with kind='line':

In [ ]:
rsamObj.plot(metrics=['median', 'mean'], kind='line'); 
# shorthand for metrics=['VLP', 'LP', 'VT', 'fratio']

Note how the max level in each 60-s window is much much higher than the median level. 

Finally, we can plot the frequency band information:

In [ ]:
rsamObj.plot(metrics='bands', kind='line');


In [ ]:
rsamObj.plot(metrics='fratio', kind='line', ylims=(-3,3)) 

## 5. RSAM on archives?

What if you have a large data archive, and want to compute RSAM for all vertical channels, for all days?

In that case, it makes sense to loop through the data one day at a time.

Here is an example using data from the Montserrat Volcano Observatory:


In [ ]:
import os
import obspy
from obspy.clients.filesystem.sds import Client as sdsclient
from flovopy.processing.sam import RSAM
SAM_DIR = os.path.expanduser('~/work/CompSci26/week10/sam_data')
os.makedirs(SAM_DIR, exist_ok=True)


In [ ]:

# Compute RSAM in 1-day chunks for multiple network-station-location-channel's
mySDSclient = sdsclient(str(SDS_ROOT))
startTime = obspy.core.UTCDateTime(2009,3,15)
endTime = obspy.core.UTCDateTime(2009,3,30)
secondsPerDay = 60 * 60 * 24
numDays = (endTime-startTime)/secondsPerDay
daytime = startTime
while daytime < endTime:
    print(f'Loading Stream data for {daytime}')
    for STA in STATIONS:
        st3 = mySDSclient.get_waveforms(NET, STA, LOC, "[SBEHCD]*Z", daytime, daytime+secondsPerDay)
        try:
            st3.merge()
        except Exception as e:
            print(f'Error occurred while merging data for {STA} at {daytime}: {e}')
            mean_samp_rate = np.array([tr.stats.sampling_rate for tr in st3]).mean()
            print(f'Average sampling rate for {STA} at {daytime}: {mean_samp_rate:.2f} Hz')
            for tr in st3:
                tr.stats.sampling_rate = mean_samp_rate
            st3.merge()

        clip_stream(st3, percentile=99.9)
        print(f'- got {len(st3)} Trace ids')
        print(f'Computing RSAM metrics for {STA} for {daytime}, and saving to CSV files')
        rsam24h = RSAM(stream=st3, sampling_interval=600)
        rsam24h.write(str(SAM_DIR), ext='csv', overwrite=True)
    daytime += secondsPerDay
del mySDSclient

In [ ]:
# Open a CSV file
os.system(f'cat {SAM_DIR}/RSAM/AV/RSAM_AV.RSO..EHZ_2009_60s.csv')

In [ ]:
import pandas as pd
files=['/Users/GlennThompson/work/CompSci26/week10/sam_data/RSAM/AV/RSAM_AV.RSO..EHZ_2009_600s.csv',
       '/Users/GlennThompson/work/CompSci26/week10/sam_data/RSAM/AV/RSAM_AV.RSO..EHZ_2009_599s.csv']
list_of_dfs = []
for f in files:
    print(f'Contents of {f}:')
    df = pd.read_csv(f)
    list_of_dfs.append(df)
master_df = pd.concat(list_of_dfs, ignore_index=True)
master_df.to_csv(files[0], index=False)

In [ ]:
# Read all the RSAM data back, and plot
rsamObj = RSAM.read(startTime, endTime, SAM_DIR=str(SAM_DIR), ext='csv', sampling_interval=600, verbose=True)
rsamObj = rsamObj.select(component='Z')
rsamObj.plot(metrics='median')

In [ ]:
REFrsamObj = rsamObj.select(id='AV.REF..EHZ')
REFrsamObj.plot(metrics=['median','fratio'])

In [ ]:
from flovopy.processing.sam_decomposition import decompose_sam
REFrsamObj = rsamObj.select(id='AV.REF..EHZ')
STA_SECONDS, LTA_SECONDS = 1200, 7200
cat = decompose_sam(REFrsamObj, metric='max', sta=STA_SECONDS, lta=LTA_SECONDS, outfile=None)

## 6. Legacy RSAM data 

### 6.1 Loading legacy RSAM data from binary files

The RSAM system was used at many observatories, and so many observatories likely have archives of RSAM binary files. But we can read these, making them Interoperable and Reusable. (Tiltmeter was saved in the same format, and so can also be read).

Next we will load 1 year of RSAM data for 8 stations recorded by the original RSAM system that was deployed in Montserrat. These data only have a 'mean' metric - it is just how they were recorded at the time.


In [ ]:
stime = obspy.core.UTCDateTime(1995,8,1,0,0,0)
etime = obspy.core.UTCDateTime(1999,12,31,23,59,59)
BINARY_DIR = DATA_ROOT / "Montserrat" / "ASN" / "RSAM" / "RSAM_1"
print(BINARY_DIR)
all_files = []
for yyyy in range(stime.year, etime.year+1):
    files = list(BINARY_DIR.glob(f'M???{yyyy}.DAT'))
    all_files.extend(files)
print(f'Found {len(all_files)} RSAM binary files')
stations = sorted(set(path.name[0:4] for path in all_files))
print(type(stations))
print('Legacy station names found:')
for index, this_station in enumerate(stations):
    print(f'{index+1:03d}: {this_station}')

These stations all date from before the SEED id convention of network.station.location.channel was used. So all that information used to be embedded into the station code. For example, MCPE, MCPN, and MCPZ are three components of MCP. And the default was 100 samples per second, and all instruments were short-period. And the network is MV. So the correct SEED ids for these are MV.MCP..EHE, MV.MCP..EHN, and MV.MCP..EHZ.

The RSAM binary file reader method (RSAM.readRSAMbinary) will attempt to convert legacy station names to SEED ids, as we will see - when the plots below appear, they will have full SEED ids.

In [ ]:
rsamObj = RSAM.readRSAMbinary(str(BINARY_DIR), stations, stime, etime, convert_legacy_ids_using_this_network='MV')
rsamObj.plot()


The only metric available from legacy RSAM binary files (created by the original RSAM system) is mean absolute data. It didn't make any other measurements. These are handled properly by the RSAM class:

In [ ]:
print(rsamObj.get_metrics())

### 6.2 Converting legacy RSAM binary files to modern RSAM CSV/Pickle files
Since we have already read the binary files into a (single) RSAM object, writing them to modern RSAM data format is as simple as:

In [ ]:
rsamObj.write(str(SAM_DIR), ext='csv')

It is now trivial to load these data next time.

### 6.3 Subsetting an RSAM object

We can see from this that the best SEED ids are MV.MGH..EHZ, MV.MLG..ELZ, MV.MLGT..EHZ, and MV.MRYT..EHZ.

So let's subset these using the select_ids() method.

In [ ]:
ids = ["MV.MGH..EHZ", "MV.MLG..ELZ", "MV.MLGT..EHZ", "MV.MRYT..EHZ"]
rsamObj2 = rsamObj.select_ids(ids=ids)
print(rsamObj2)
rsamObj2.plot()

### 6.4 Downsample 

To get a proxy for overall seismicity level, we will sometimes want to smooth RSAM data. So let's smooth from 1 minute sampling rate to 1 day.

In [ ]:
rsamObj5 = rsamObj4.downsample(new_sampling_interval=86400) 
rsamObj5.plot()


This gives a crude overview of the overall seismicity level from August 1995 through December 1999. 
Questions:
1. When does it look most active to you?
2. Does it depend on which station you look at?

## 7. RSAM data processing and analysis

### 7.1 read and plot

Next we will:
- (re-)read (from disk) the RSAM data from 1996/08/01 to 1996/08/05 for select SEED ids
- plot the data. By default, the plot() method will convert RSAM dataframes into an ObsPy Stream object, so it can be plotted in a familiar way.

In [ ]:
startt = obspy.core.UTCDateTime(1996,8,2)
endt = obspy.core.UTCDateTime(1996,8,6)
rsamObj3 = RSAM.read(startt, endt, SAM_DIR=str(SAM_DIR), ext='csv')
rsamObj4 = rsamObj3.select(component='Z')
rsamObj4.plot()   

These are remarkable cycles in RSAM. They appear to be about 4-6 hours apart. This is a phenomenon called "banded tremor". During these tremor bands, visual observations indicated that the lava dome was extruding at particularly high rates (up to 20m^3 was one estimate I heard), and at the peak of each cycle there was often ash venting. I proposed that the tremor bands were indicated of pressure cycles within the conduit - but caused by what? 
One suggestion is that the magma rises up the conduit in a stick-slip fashion. Basically, it gets stuck for a while, as the pressure builds below, and then shear fractures, allowing magma to suddenly extrude very quickly. 

Can we use some ObsPy STA/LTA detection tools to detect these tremor bands, in the same way we normally detect much shorter transient events, but just with longer STA/LTA settings? Let us try first on a single NSLC. This is based on examples at https://docs.obspy.org/tutorial/code_snippets/trigger_tutorial.html, except we use longer STA and LTA time windows (15 and 100 minutes respectively), and we add a despiking step which attempts to remove transient events lasting a minute or less from the data before running the STA/LTA:


### 7.3 Tremor band detection with ObsPy trigger methods

#### 7.3.1 Single channel detection

In [ ]:
from obspy.signal.trigger import plot_trigger, classic_sta_lta, recursive_sta_lta

st2 = rsamObj2.to_stream()

sta_minutes = 15
lta_minutes = 100
threshON = 0.8
threshOFF = 0.5

for tr in st2:
    print(tr)
    sampling_interval_minutes = tr.stats.delta/60
    cft = recursive_sta_lta(tr.data, int(sta_minutes / sampling_interval_minutes), int(lta_minutes / sampling_interval_minutes))

    plot_trigger(tr, cft, threshON, threshOFF)

That seems to work quite well. Now let us try an event detector that uses several NSLC at once.

#### 7.3.2 Multi-channel detection

In [ ]:
from obspy.signal.trigger import coincidence_trigger
from pprint import pprint
import numpy as np

threshStations = 3
triggerMethod = 'recstalta'
maxTriggerSecs = 2*lta_minutes*60

trig = coincidence_trigger(triggerMethod, threshON, threshOFF, st2, threshStations, sta=sta_minutes*60, lta=lta_minutes*60, 
                           max_trigger_length=maxTriggerSecs, delete_long_trigger=True)

In [ ]:
from flovopy.processing.sam_decomposition import decompose_sam
STA_SECONDS, LTA_SECONDS = 1200, 7200
cat = decompose_sam(rsamObj2, metric='mean', sta=STA_SECONDS, lta=LTA_SECONDS, outfile=None)

In [ ]:
import vsmTools
import importlib
importlib.reload(vsmTools)
bandedTremorCat = vsmTools.triggers2catalog(trig, triggerMethod, threshON, threshOFF, sta_minutes*60, lta_minutes*60, maxTriggerSecs)

In [ ]:
bandedTremorDF = bandedTremorCat.to_dataframe()
print(bandedTremorDF)

In [ ]:
import pandas as pd
bandedTremorCat.plot_eventrate(binsize=pd.Timedelta(days=1))

In [ ]:
bandedTremorDF.plot.scatter(x='datetime', y='duration', rot=90)